# 1 — Attribution Run

This notebook is the **data-generation pipeline** for the circuit-tracing experiment.
It is designed to run on a GPU machine (Runpod) and produces a self-contained results directory
under `my_work/results/runs/<run_id>/`.

The only external input is `my_work/data/questions.json`. To run a different question set,
change that file and re-run this notebook — no notebook code changes are needed.

The experiment follows these sequential steps:

1. **Baseline sweep** — record the model's unaided `True`/`False` probabilities for every (question, template) pair.
2. **Attribution graph construction** — for each question under the active template, build a feature attribution graph.
3. **Top feature extraction** — rank and save the most influential active features per question.
4. **Cross-question recurrence** — identify features that appear across multiple questions.
5. **Circuit interventions** — amplify and ablate recurring features to test causal influence.
6. **Persist all outputs** — write manifest, tables, raw graph files (.pt), and viewer-ready JSON exports.

After this notebook completes, commit and push the run directory to git.
The analysis notebook (`2_analyse_results.ipynb`) then loads those files on your local machine,
no GPU or model required.

---

## Section 0 — Environment Setup

Three cells handle environment setup in order:

1. **Dependency bootstrap** — installs the `circuit-tracer` package. Run once on a fresh pod, then restart the kernel.
2. **Runtime and Hugging Face setup** — locates the repo root, validates packages, pins the HF stack, loads `.env`, configures caches.
3. **Sanity check** — prints GPU availability, torch version, and environment variables.

In [1]:
# Run this cell once on a fresh pod or new kernel, then restart the kernel.
# After restarting, skip this cell and continue from the next one.

# %pip install --no-cache-dir -e /workspace/thesis_circuit_breaker

In [1]:
#@title Runtime + Hugging Face Setup (Local/Runpod) { display-mode: "form" }

import os
import sys
from pathlib import Path

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None
print("Configuring local/Runpod environment...")


def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path
    return None


_root = _find_repo_root()
if _root is not None:
    root_str = str(_root)
    if root_str not in sys.path:
        sys.path.insert(0, root_str)
    print(f"Repo root on sys.path: {_root}")

    import importlib.util, subprocess
    required_modules = ["safetensors", "transformers", "tokenizers", "nnsight", "transformer_lens"]
    missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
    if missing:
        raise RuntimeError(f"Missing dependencies: {missing}. Install with: pip install -e {_root} then restart kernel.")

    from importlib.metadata import version as _pkg_version

    def _parse(v):
        core = v.split("+", 1)[0].strip()
        parts = core.split(".")
        nums = []
        for p in parts[:3]:
            digits = "".join(ch for ch in p if ch.isdigit())
            nums.append(int(digits) if digits else 0)
        while len(nums) < 3:
            nums.append(0)
        return tuple(nums)

    def _lt(a, b): return _parse(a) < _parse(b)
    def _gt(a, b): return _parse(a) > _parse(b)
    def _gte(v, minimum): return not _lt(v, minimum)

    def _pin_hf_stack():
        try:
            hf_hub_v = _pkg_version("huggingface_hub")
        except Exception:
            hf_hub_v = ""
        try:
            tf_v = _pkg_version("transformers")
        except Exception:
            tf_v = ""
        needs_pin = (hf_hub_v and _gte(hf_hub_v, "1.0.0")) or (
            tf_v and (_lt(tf_v, "4.56.0") or _gt(tf_v, "4.57.3"))
        )
        if needs_pin:
            print("Pinning huggingface_hub + transformers to repo-compatible versions...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
                "huggingface_hub<1.0.0", "transformers>=4.56.0,<=4.57.3"])
            raise RuntimeError("Pinned huggingface_hub/transformers. Restart the kernel, then rerun from top.")

    _pin_hf_stack()
else:
    print("Could not locate circuit_tracer repo. Set CT_REPO_DIR or clone into /workspace.")

try:
    from typing_extensions import TypeIs  # type: ignore[attr-defined]
except Exception:
    print("typing_extensions too old; upgrading...")
    %pip install -q --no-cache-dir "typing_extensions>=4.12.2"
    raise RuntimeError("Installed typing_extensions>=4.12.2. Restart kernel and rerun.")

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

if load_dotenv is not None and _root is not None:
    _env_file = _root / ".env"
    if _env_file.is_file():
        load_dotenv(_env_file)
        print(f"Loaded {_env_file}")

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if hf_token and not os.environ.get("HUGGINGFACE_HUB_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token

if IN_RUNPOD:
    persistent_root = Path(os.environ.get("RUNPOD_PERSISTENT_ROOT", "/workspace")).resolve()
    hf_home = persistent_root / "hf"
    cache_dirs = {
        "HF_HOME": hf_home,
        "HUGGINGFACE_HUB_CACHE": hf_home / "hub",
        "TRANSFORMERS_CACHE": hf_home / "transformers",
        "HF_DATASETS_CACHE": hf_home / "datasets",
        "TORCH_HOME": persistent_root / "torch",
    }
    for key, path in cache_dirs.items():
        path.mkdir(parents=True, exist_ok=True)
        os.environ[key] = str(path)
    print(f"Runpod persistent storage configured under {persistent_root}")

from huggingface_hub import get_token
if get_token() is None:
    print("No Hugging Face token found. google/gemma-2-2b is a gated model. "
          "Accept the licence at https://huggingface.co/google/gemma-2-2b and set HF_TOKEN.")
else:
    print("Hugging Face token detected.")

Configuring local/Runpod environment...
Repo root on sys.path: /workspace/thesis_circuit_breaker
Runpod persistent storage configured under /workspace
Hugging Face token detected.


In [2]:
import os, torch

print(f'torch version : {torch.__version__}')
print(f'cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu           : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: no GPU detected — inference will be very slow on CPU.')

for key in ['HF_HOME', 'HUGGINGFACE_HUB_CACHE', 'TRANSFORMERS_CACHE', 'HF_DATASETS_CACHE', 'TORCH_HOME']:
    print(f'{key}={os.environ.get(key, "<unset>")}')

print('HF token set:', bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')))

torch version : 2.4.1+cu124
cuda available: True
gpu           : NVIDIA RTX A4500
HF_HOME=/workspace/hf
HUGGINGFACE_HUB_CACHE=/workspace/hf/hub
TRANSFORMERS_CACHE=/workspace/hf/transformers
HF_DATASETS_CACHE=/workspace/hf/datasets
TORCH_HOME=/workspace/torch
HF token set: True


---

## Imports

All Python imports used throughout this notebook are consolidated here.
Run this cell after environment setup. If an import fails, the dependency installation step above did not complete successfully.

In [3]:
import csv
import hashlib
import json
import re
import subprocess
import sys
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

import torch
from IPython.display import IFrame, display

from circuit_tracer import Graph, ReplacementModel, attribute
from circuit_tracer.attribution.targets import CustomTarget
from circuit_tracer.frontend.local_server import serve
from circuit_tracer.utils import create_graph_files, get_default_device
from circuit_tracer.utils.demo_utils import (
    cleanup_cuda,
    display_top_features_comparison,
    get_top_features,
    get_unembed_vecs,
)

print('All imports successful.')

/workspace/venvs/ct/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


All imports successful.


---

## Question Configuration

All questions and prompt templates are defined in `my_work/data/questions.json`.
Edit that file to change the question set — no notebook changes are required.

A snapshot of the config is saved alongside the run artifacts so each run is
fully reproducible even if `questions.json` is later modified.

In [28]:
QUESTION_CONFIG_FILENAME = 'questions_texas_capital_tf.json'  # change this once (e.g. 'questions_math.json')

def _find_config_path(config_filename: str = QUESTION_CONFIG_FILENAME) -> Path:
    candidates = []

    if '_root' in globals() and _root is not None:  # type: ignore[name-defined]
        candidates.append(_root / 'my_work' / 'data' / config_filename)  # type: ignore[name-defined]

    repo_override = os.environ.get('CT_REPO_DIR')
    if repo_override:
        candidates.append(Path(repo_override).expanduser().resolve() / 'my_work' / 'data' / config_filename)

    cwd = Path.cwd().resolve()
    for parent in [cwd, *cwd.parents]:
        candidates.extend([
            parent / 'my_work' / 'data' / config_filename,
            parent / 'data' / config_filename,
        ])

    for root in [Path('/workspace'), Path('/root'), Path('/home')]:
        if not root.is_dir():
            continue
        candidates.append(root / 'my_work' / 'data' / config_filename)
        try:
            for child in root.iterdir():
                if child.is_dir():
                    candidates.append(child / 'my_work' / 'data' / config_filename)
                    candidates.append(child / 'data' / config_filename)
        except Exception:
            pass

    seen = set()
    for cand in candidates:
        if cand in seen:
            continue
        seen.add(cand)
        if cand.is_file():
            return cand

    raise FileNotFoundError(
        f'Could not locate my_work/data/{config_filename}. '
        f'Current working directory is: {Path.cwd().resolve()}. '
        'Set CT_REPO_DIR to your repository path and ensure the file exists.'
    )


CONFIG_PATH = _find_config_path()
cfg         = json.loads(CONFIG_PATH.read_text())
TEMPLATES        = cfg['templates']
QUESTIONS        = cfg['questions']
ACTIVE_TEMPLATE  = cfg['metadata']['active_template']

NOTEBOOK_DIR = CONFIG_PATH.parent.parent / 'notebooks'

print(f'Config loaded from : {CONFIG_PATH}')
print(f'Templates defined  : {list(TEMPLATES.keys())}')
print(f'Active template    : {ACTIVE_TEMPLATE}')
print(f'Questions loaded   : {len(QUESTIONS)}')
print()
for q in QUESTIONS:
    label_str = 'TRUE ' if q['label'] else 'FALSE'
    print(f"  [{label_str}] {q['id']}: {q['statement'][:80]}")

Config loaded from : /workspace/thesis_circuit_breaker/my_work/data/questions_texas_capital_tf.json
Templates defined  : ['T1']
Active template    : T1
Questions loaded   : 5

  [TRUE ] TX-01: Fact: the capital of the state containing Dallas is Austin.
  [FALSE] TX-02: Fact: the capital of the state containing Dallas is Houston.
  [TRUE ] TX-03: Dallas is in Texas.
  [TRUE ] TX-04: Austin is the capital of Texas.
  [FALSE] TX-05: Texas has no capital city.


---

## Run Directory

The run directory is **`my_work/results/runs/<run_id>/`**. The `run_id` is **deterministic**:
it is derived from a SHA-256 hash of a JSON snapshot that combines the full question bank
(`cfg` from `questions.json`) plus model and attribution parameters used in this notebook
(see the code cell). The same inputs produce the same `run_id` on any day, so partial
`graphs_pt/*.pt` files line up with the resume/skip logic.

On the first run of the setup cell, `run_config_snapshot.json` is written **immediately** under
that path (before model load). The manifest is written later and adds git SHA and wall-clock time.

Directory structure:

```
my_work/results/runs/<run_id>/
    run_config_snapshot.json <- question bank + model + attribution params; basis of run_id hash
    manifest.json            <- model/config/git/timestamp and intervention hyperparams
    tables/
        baseline_results.csv
        top_features.csv
        feature_frequency.csv
        interventions.csv
        summary_metrics.json
    graphs_pt/               <- raw attribution graph files (.pt)
    graphs_json/             <- viewer-ready JSON exports
```

In [29]:
def _run_slug(s: str, max_len: int = 32) -> str:
    t = re.sub(r'[^\w\-]+', '_', s, flags=re.ASCII).strip('_')
    return (t[:max_len].strip('_') or 'run')


MODEL_NAME      = 'google/gemma-2-2b'
TRANSCODER_NAME = 'gemma'
BACKEND         = 'transformerlens'
MODEL_DTYPE     = 'bfloat16'

ATTR_BATCH_SIZE        = 128
ATTR_MAX_FEATURE_NODES = 8192
ATTR_OFFLOAD           = 'disk'
ATTR_VERBOSE           = True

FINGERPRINT_VERSION = 1

_desc = (cfg.get('metadata') or {}).get('description', '') or 'dataset'
dataset_tag = _run_slug(_desc.split('—')[0].strip() if '—' in _desc else _desc)

run_config_snapshot = {
    'fingerprint_version': FINGERPRINT_VERSION,
    'question_bank':       cfg,
    'model': {
        'name':        MODEL_NAME,
        'transcoder':  TRANSCODER_NAME,
        'backend':     BACKEND,
        'dtype':       MODEL_DTYPE,
    },
    'attribution': {
        'batch_size':        ATTR_BATCH_SIZE,
        'max_feature_nodes': ATTR_MAX_FEATURE_NODES,
        'offload':           ATTR_OFFLOAD,
        'verbose':           ATTR_VERBOSE,
    },
}

SNAPSHOT_TEXT = json.dumps(
    run_config_snapshot,
    sort_keys=True,
    ensure_ascii=False,
    indent=2,
)
CONFIG_HASH = hashlib.sha256(SNAPSHOT_TEXT.encode('utf-8')).hexdigest()[:12]

model_tag = 'gemma2-2b'
run_id    = f'{model_tag}_{dataset_tag}_{CONFIG_HASH}'

_results_base = _root / 'my_work' / 'results' / 'runs' if _root else Path.cwd() / 'results'
RESULTS_ROOT    = _results_base / run_id
TABLES_DIR      = RESULTS_ROOT / 'tables'
GRAPHS_PT_DIR   = RESULTS_ROOT / 'graphs_pt'
GRAPHS_JSON_DIR = RESULTS_ROOT / 'graphs_json'

for d in [TABLES_DIR, GRAPHS_PT_DIR, GRAPHS_JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

(RESULTS_ROOT / 'run_config_snapshot.json').write_text(SNAPSHOT_TEXT)

print(f'run_id       : {run_id}')
print(f'Results root : {RESULTS_ROOT}')
print(f'Wrote        : {RESULTS_ROOT / "run_config_snapshot.json"} (this exact JSON is the hash input)')

run_id       : gemma2-2b_Phase_1_mini_bank_Texas-capital_ca8e1bf8cf68
Results root : /workspace/thesis_circuit_breaker/my_work/results/runs/gemma2-2b_Phase_1_mini_bank_Texas-capital_ca8e1bf8cf68
Wrote        : /workspace/thesis_circuit_breaker/my_work/results/runs/gemma2-2b_Phase_1_mini_bank_Texas-capital_ca8e1bf8cf68/run_config_snapshot.json (this exact JSON is the hash input)


---

## Model Loading

`ReplacementModel.from_pretrained` loads the base language model together with its pre-trained
cross-layer transcoder (CLT) weights. The replacement model is the version of Gemma-2-2b in
which MLP outputs are approximated by sparse transcoder features — this is the model on which
attribution graphs are built.

The first run downloads and caches model weights. Subsequent runs reuse the cache.

In [6]:
device = get_default_device()
print(f'Loading model on {device} (cuda > mps > cpu)')

_torch_dtype = {'bfloat16': torch.bfloat16, 'float16': torch.float16, 'float32': torch.float32}[MODEL_DTYPE]

model = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_NAME,
    dtype=_torch_dtype,
    backend=BACKEND,
)
tokenizer = model.tokenizer
print(f'Model loaded: {MODEL_NAME}')

Loading model on cuda (cuda > mps > cpu)


Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
Model loaded: google/gemma-2-2b


---

## Token Sanity Check

We must confirm that `True` and `False` each tokenise to exactly one token. The attribution
target is defined as `logit(True) - logit(False)`. If either word maps to multiple tokens,
the target vector is wrong.

We check space-prefixed forms because that is the expected format at the answer position
in the prompt (Gemma uses a special marker for word-initial tokens).

In [7]:
TOKEN_TRUE_STR  = ' True'
TOKEN_FALSE_STR = ' False'

ids_true  = tokenizer.encode(TOKEN_TRUE_STR,  add_special_tokens=False)
ids_false = tokenizer.encode(TOKEN_FALSE_STR, add_special_tokens=False)

assert len(ids_true)  == 1, f"' True' tokenises to {len(ids_true)} tokens: {ids_true}"
assert len(ids_false) == 1, f"' False' tokenises to {len(ids_false)} tokens: {ids_false}"

IDX_TRUE  = ids_true[0]
IDX_FALSE = ids_false[0]

print(f"Token '{TOKEN_TRUE_STR}'  -> vocab ID {IDX_TRUE}  (decoded: '{tokenizer.decode([IDX_TRUE])}')")
print(f"Token '{TOKEN_FALSE_STR}' -> vocab ID {IDX_FALSE} (decoded: '{tokenizer.decode([IDX_FALSE])}')")
print('Sanity check passed.')

Token ' True'  -> vocab ID 5569  (decoded: ' True')
Token ' False' -> vocab ID 7662 (decoded: ' False')
Sanity check passed.


In [27]:
# Pick one question (adjust id if needed)
qid = "TX-01"  # false case: Houston
q = next(x for x in QUESTIONS if x["id"] == qid)
prompt = TEMPLATES[ACTIVE_TEMPLATE].replace("{S}", q["statement"])

prompt = "Statement: Fact: the capital of the state containing Dallas is Austin. Answer:"

print("Prompt:")
print(prompt)
print("-" * 80)

# Last-token logits
input_ids = model.ensure_tokenized(prompt)
with torch.no_grad():
    logits, _ = model.get_activations(input_ids)

last = logits.squeeze(0)[-1].float()
probs = torch.softmax(last, dim=-1)

# Top-5 next-token predictions
print("Top-5 next tokens:")
topk = torch.topk(probs, k=5)
for rank, (tid, p) in enumerate(zip(topk.indices.tolist(), topk.values.tolist()), start=1):
    tok = tokenizer.decode([tid])
    logit = last[tid].item()
    print(f"{rank}. id={tid:6d}  token={tok!r:20}  prob={p:.6e}  logit={logit:+.4f}")

# Candidate answer tokens to compare
cands = ["_True", "_False", " True", " False", "True", "False", "\nTrue", "\nFalse", " Yes", " No"]
ids = {t: tokenizer.encode(t, add_special_tokens=False) for t in cands}

print("\nTokenization:")
for t, tok_ids in ids.items():
    print(f"{t!r:10} -> {tok_ids}")

print("\nScores (single-token candidates only):")
rows = []
for t, tok_ids in ids.items():
    if len(tok_ids) == 1:
        tid = tok_ids[0]
        rows.append((t, tid, last[tid].item(), probs[tid].item()))

rows = sorted(rows, key=lambda x: x[2], reverse=True)
for t, tid, logit, prob in rows:
    print(f"{t!r:10} id={tid:6d}  logit={logit:+.4f}  prob={prob:.6e}")

# Your current decision metric
if len(ids[" True"]) == 1 and len(ids[" False"]) == 1:
    it = ids[" True"][0]
    iff = ids[" False"][0]
    margin = last[it].item() - last[iff].item()
    print(f"\nCurrent margin (' True' - ' False') = {margin:+.4f}")

Prompt:
Statement: Fact: the capital of the state containing Dallas is Austin. Answer:
--------------------------------------------------------------------------------
Top-5 next tokens:
1. id=  5569  token=' True'               prob=1.103903e-01  logit=+27.1250
2. id= 21427  token=' Fact'               prob=8.597206e-02  logit=+26.8750
3. id=   586  token=' A'                  prob=6.695511e-02  logit=+26.6250
4. id=  7662  token=' False'              prob=5.214470e-02  logit=+26.3750
5. id=   714  token=' The'                prob=4.601753e-02  logit=+26.2500

Tokenization:
'_True'    -> [235298, 5036]
'_False'   -> [235298, 8393]
' True'    -> [5569]
' False'   -> [7662]
'True'     -> [5036]
'False'    -> [8393]
'\nTrue'   -> [108, 5036]
'\nFalse'  -> [108, 8393]
' Yes'     -> [6287]
' No'      -> [1307]

Scores (single-token candidates only):
' True'    id=  5569  logit=+27.1250  prob=1.103903e-01
' False'   id=  7662  logit=+26.3750  prob=5.214470e-02
' Yes'     id=  6287  logit=+2

---

## Section 1 — Baseline Sweep

The baseline sweep measures the model's unaided preference for `True` vs `False` across all
questions and all prompt templates. No attribution or intervention is performed here.

For each (question, template) combination:
1. Format the prompt.
2. Run a forward pass to obtain output logits.
3. Read `logit(True)` and `logit(False)` at the last token position.
4. Compute `margin = logit(True) - logit(False)`.

A positive margin means the model predicts `True`; negative means `False`.

In [9]:
def format_prompt(question: dict, template_key: str) -> str:
    return TEMPLATES[template_key].replace('{S}', question['statement'])


baseline_records = []

for q in QUESTIONS:
    for tkey in TEMPLATES:
        prompt    = format_prompt(q, tkey)
        input_ids = model.ensure_tokenized(prompt)

        with torch.no_grad():
            logits, _ = model.get_activations(input_ids)

        last    = logits.squeeze(0)[-1]
        l_true  = last[IDX_TRUE].item()
        l_false = last[IDX_FALSE].item()
        margin  = l_true - l_false
        predicted = margin > 0
        correct   = predicted == q['label']

        baseline_records.append({
            'id':           q['id'],
            'template':     tkey,
            'label':        q['label'],
            'logit_true':   round(l_true,  4),
            'logit_false':  round(l_false, 4),
            'margin':       round(margin,  4),
            'predicted':    predicted,
            'correct':      correct,
        })
        cleanup_cuda()

csv_path = TABLES_DIR / 'baseline_results.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(baseline_records[0].keys()))
    writer.writeheader()
    writer.writerows(baseline_records)

print(f'Baseline sweep complete. {len(baseline_records)} runs saved to {csv_path}')

In [10]:
print(f"{'ID':<8} {'Tmpl':<5} {'Label':<6} {'Margin':>8} {'Pred':<6} {'OK'}")
print('-' * 46)
for r in baseline_records:
    label_str = 'True ' if r['label'] else 'False'
    pred_str  = 'True ' if r['predicted'] else 'False'
    ok_str    = 'OK' if r['correct'] else 'WRONG'
    print(f"{r['id']:<8} {r['template']:<5} {label_str:<6} {r['margin']:>8.4f} {pred_str:<6} {ok_str}")

print()
for tkey in TEMPLATES:
    recs = [r for r in baseline_records if r['template'] == tkey]
    n_correct = sum(1 for r in recs if r['correct'])
    print(f'Accuracy {tkey}: {n_correct}/{len(recs)} ({100 * n_correct / len(recs):.0f}%)')

In [11]:
[x for x in recs if x['predicted'] == False]

### 1.1 Raw-Model Baseline (Original Gemma)

To check whether the replacement model's baseline behavior matches the original model,
this subsection runs the same `(question, template)` sweep on **raw Gemma** loaded via
`transformers`.

Outputs written to the run table directory:

- `baseline_results_raw.csv` — raw-model logits/margins/predictions
- `baseline_comparison.csv` — row-wise comparison of replacement vs raw results

By default, `RAW_DEVICE='cpu'` to avoid GPU OOM while the replacement model is loaded.
If you have enough free VRAM and want speed, set `RAW_DEVICE='cuda'`.

Prompts are tokenized with `add_special_tokens=True` so the input matches
`model.ensure_tokenized(...)` in the replacement baseline (e.g. leading BOS for Gemma).


In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer

RAW_MODEL_NAME = MODEL_NAME
RAW_DEVICE = 'cuda'  # 'cpu' (safe) or 'cuda' (faster, needs free VRAM)
RAW_DTYPE = torch.bfloat16 if RAW_DEVICE == 'cuda' else torch.float32

print(f'Loading raw baseline model: {RAW_MODEL_NAME} on {RAW_DEVICE} ({RAW_DTYPE})')
raw_tokenizer = AutoTokenizer.from_pretrained(RAW_MODEL_NAME)
raw_model = AutoModelForCausalLM.from_pretrained(
    RAW_MODEL_NAME,
    torch_dtype=RAW_DTYPE,
)
raw_model.to(RAW_DEVICE)
raw_model.eval()

raw_ids_true = raw_tokenizer.encode(TOKEN_TRUE_STR, add_special_tokens=False)
raw_ids_false = raw_tokenizer.encode(TOKEN_FALSE_STR, add_special_tokens=False)
assert len(raw_ids_true) == 1, f"Raw tokenizer: {TOKEN_TRUE_STR!r} -> {raw_ids_true}"
assert len(raw_ids_false) == 1, f"Raw tokenizer: {TOKEN_FALSE_STR!r} -> {raw_ids_false}"
RAW_IDX_TRUE = raw_ids_true[0]
RAW_IDX_FALSE = raw_ids_false[0]

raw_records = []
for q in QUESTIONS:
    for tkey in TEMPLATES:
        prompt = format_prompt(q, tkey)
        enc = raw_tokenizer(prompt, return_tensors='pt', add_special_tokens=True)
        enc = {k: v.to(RAW_DEVICE) for k, v in enc.items()}

        with torch.no_grad():
            out = raw_model(**enc)
        last = out.logits[0, -1]

        l_true = last[RAW_IDX_TRUE].item()
        l_false = last[RAW_IDX_FALSE].item()
        margin = l_true - l_false
        pred = margin > 0

        raw_records.append({
            'id': q['id'],
            'template': tkey,
            'label': q['label'],
            'logit_true_raw': round(l_true, 4),
            'logit_false_raw': round(l_false, 4),
            'margin_raw': round(margin, 4),
            'predicted_raw': pred,
            'correct_raw': pred == q['label'],
        })

raw_csv = TABLES_DIR / 'baseline_results_raw.csv'
with open(raw_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(raw_records[0].keys()))
    writer.writeheader()
    writer.writerows(raw_records)

print(f'Raw baseline complete. {len(raw_records)} rows saved to {raw_csv}')

del raw_model
cleanup_cuda()


In [13]:
# Row-wise comparison: replacement baseline vs raw baseline
raw_by_key = {(r['id'], r['template']): r for r in raw_records}
comparison_records = []

for r in baseline_records:
    key = (r['id'], r['template'])
    rr = raw_by_key[key]
    comparison_records.append({
        'id': r['id'],
        'template': r['template'],
        'label': r['label'],
        'margin_replacement': r['margin'],
        'margin_raw': rr['margin_raw'],
        'predicted_replacement': r['predicted'],
        'predicted_raw': rr['predicted_raw'],
        'agree_prediction': r['predicted'] == rr['predicted_raw'],
        'delta_margin_repl_minus_raw': round(r['margin'] - rr['margin_raw'], 4),
    })

cmp_csv = TABLES_DIR / 'baseline_comparison.csv'
with open(cmp_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(comparison_records[0].keys()))
    writer.writeheader()
    writer.writerows(comparison_records)

n = len(comparison_records)
agree = sum(1 for x in comparison_records if x['agree_prediction'])
print(f'Comparison saved to {cmp_csv}')
print(f'Prediction agreement (replacement vs raw): {agree}/{n} ({100*agree/n:.1f}%)')

print('\nRows where predictions disagree:')
for x in comparison_records:
    if not x['agree_prediction']:
        pr = 'True' if x['predicted_replacement'] else 'False'
        po = 'True' if x['predicted_raw'] else 'False'
        print(f"  {x['id']} {x['template']}: repl={pr}, raw={po}, "
              f"m_repl={x['margin_replacement']:+.4f}, m_raw={x['margin_raw']:+.4f}")


In [14]:
comparison_df = pd.DataFrame(comparison_records)
comparison_df.sort_values(by='delta_margin_repl_minus_raw', ascending=False)

In [15]:
# Quick alignment diagnostic: replacement vs raw tokenization + BOS behavior

qid = QUESTIONS[0]["id"]          # e.g. TI-01
tkey = ACTIVE_TEMPLATE            # use same template as attribution
q = next(x for x in QUESTIONS if x["id"] == qid)
prompt = format_prompt(q, tkey)

print("Prompt:")
print(prompt)
print("-" * 80)

# Replacement model tokenization
rep_ids = model.ensure_tokenized(prompt).squeeze(0).tolist()
print("Replacement token count:", len(rep_ids))
print("Replacement first 12 ids:", rep_ids[:12])

# Raw tokenizer without special tokens
raw_no = raw_tokenizer(prompt, return_tensors="pt", add_special_tokens=False)["input_ids"][0].tolist()
print("\nRaw (add_special_tokens=False) count:", len(raw_no))
print("Raw no-special first 12 ids:", raw_no[:12])

# Raw tokenizer with special tokens
raw_yes = raw_tokenizer(prompt, return_tensors="pt", add_special_tokens=True)["input_ids"][0].tolist()
print("\nRaw (add_special_tokens=True) count:", len(raw_yes))
print("Raw with-special first 12 ids:", raw_yes[:12])

print("\nDoes replacement match raw no-special? ", rep_ids == raw_no)
print("Does replacement match raw with-special?", rep_ids == raw_yes)

# Also verify True/False token IDs are same across tokenizers
rep_true = model.tokenizer.encode(" True", add_special_tokens=False)
rep_false = model.tokenizer.encode(" False", add_special_tokens=False)
raw_true = raw_tokenizer.encode(" True", add_special_tokens=False)
raw_false = raw_tokenizer.encode(" False", add_special_tokens=False)

print("\nToken IDs:")
print("rep ' True' :", rep_true, " | raw ' True' :", raw_true)
print("rep ' False':", rep_false, " | raw ' False':", raw_false)

---

## Section 2 — Attribution Graph Construction

An attribution graph describes which internal model components contributed to the model's
output preference and by how much, for a specific prompt and a chosen output direction.

**Attribution target: `logit(True) - logit(False)`**

We attribute to the **direction** `logit(True) - logit(False)` in the model's output space.
This is a `CustomTarget` built as the difference of the two unembedding vectors, weighted by
the absolute gap in their softmax probabilities.

Features with positive influence push the model toward `True` *over* `False`. Using a
difference direction is more diagnostic than attributing to a single output token because
it directly targets the binary classification decision.

**Compute cost and storage**

Attribution is the most expensive step. Each run processes up to `max_feature_nodes=8192`
active transcoder features. `offload='disk'` reduces GPU memory pressure. Raw graphs are
saved as `.pt` files for later reloading without re-running attribution.

**Resuming and skipping cached graphs**

After a successful question, its graph is written to `graphs_pt/<qid>_<active_template>.pt`.
On the next run of the attribution cell, if that file already exists, the cell **loads** it with
`Graph.from_pt` (CPU) and **skips** `attribute()` for that question. Set `FORCE_REATTR = True`
in the code cell to recompute all questions. If you change `questions.json`, model weights, or
attribution parameters, delete the stale `.pt` files (or use a new run directory) so you do not
mix old graphs with a new config.

In [16]:
def build_true_false_target(model, prompt: str, idx_true: int, idx_false: int, backend: str) -> CustomTarget:
    """Build a CustomTarget for the direction logit(True) - logit(False).

    The attribution vector is the difference of the unembedding columns for the two answer tokens.
    The weight is the absolute probability gap, floored at 1e-6 to avoid a zero-weight target
    when probabilities are almost equal.
    """
    vec_true, vec_false = get_unembed_vecs(model, [idx_true, idx_false], backend)
    diff_vec = vec_true - vec_false

    input_ids = model.ensure_tokenized(prompt)
    with torch.no_grad():
        logits, _ = model.get_activations(input_ids)
    probs     = torch.softmax(logits.squeeze(0)[-1].float(), dim=-1)
    diff_prob = max((probs[idx_true] - probs[idx_false]).abs().item(), 1e-6)

    return CustomTarget(token_str='logit(True)-logit(False)', prob=diff_prob, vec=diff_vec)

In [17]:
ATTR_KWARGS = dict(
    batch_size=ATTR_BATCH_SIZE,
    max_feature_nodes=ATTR_MAX_FEATURE_NODES,
    offload=ATTR_OFFLOAD,
    verbose=ATTR_VERBOSE,
)

FORCE_REATTR = False

graphs: dict = {}

for q in QUESTIONS:
    qid        = q['id']
    prompt     = format_prompt(q, ACTIVE_TEMPLATE)
    graph_path = GRAPHS_PT_DIR / f'{qid}_{ACTIVE_TEMPLATE}.pt'

    if (not FORCE_REATTR) and graph_path.is_file():
        graph = Graph.from_pt(str(graph_path), map_location='cpu')
        graphs[qid] = graph
        print(f"\n--- {qid} (label={q['label']}) — SKIP (loaded from cache) ---")
        print(f'{qid}: {graph.active_features.shape[0]} active features <- {graph_path.name}')
        cleanup_cuda()
        continue

    target = build_true_false_target(model, prompt, IDX_TRUE, IDX_FALSE, backend=BACKEND)

    print(f"\n--- Attributing {qid} (label={q['label']}) ---")
    graph = attribute(prompt=prompt, model=model, attribution_targets=[target], **ATTR_KWARGS)

    graphs[qid] = graph
    graph.to_pt(str(graph_path))

    print(f'{qid}: {graph.active_features.shape[0]} active features -> saved {graph_path.name}')
    cleanup_cuda()

print(f'\nAttribution complete. {len(graphs)} graphs in memory; files under {GRAPHS_PT_DIR}')

---

## Section 3 — Top Feature Extraction

`get_top_features` ranks all active features by their total multi-hop influence on the attribution
target, accounting for both direct effects on the output and indirect effects mediated through
chains of other features.

We extract the top `TOP_N` features per question. Each feature is identified by a
`(layer, position, feature_index)` tuple. Positive score means the feature pushes toward `True`;
negative means it pushes toward `False`.

**Note on score magnitudes**

Influence scores are weighted by the CustomTarget's `prob` field, which equals
`abs(p(True) - p(False))`. When the probability gap is small (e.g. 0.005), all scores are
scaled down proportionally and may display as `0.0000` at 4 decimal places. The relative ranking
within a question remains valid. For cross-question comparisons, use the `score_norm` column,
which rescales each question's scores to the range `[-1, 1]`.

In [18]:
TOP_N = 20

top_features_by_question: dict = {}
top_scores_by_question:   dict = {}
top_features_records = []

for qid, graph in graphs.items():
    features, scores = get_top_features(graph, n=TOP_N)
    top_features_by_question[qid] = features
    top_scores_by_question[qid]   = scores

    max_abs = max((abs(s) for s in scores), default=1e-9)
    for rank, (feat, score) in enumerate(zip(features, scores), start=1):
        layer, pos, feat_idx = feat
        top_features_records.append({
            'run_id':      run_id,
            'qid':         qid,
            'rank':        rank,
            'layer':       layer,
            'position':    pos,
            'feature_idx': feat_idx,
            'score':       round(score, 8),
            'score_norm':  round(score / max_abs if max_abs > 0 else 0.0, 6),
            'sign':        'pos' if score >= 0 else 'neg',
        })

tf_csv = TABLES_DIR / 'top_features.csv'
with open(tf_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(top_features_records[0].keys()))
    writer.writeheader()
    writer.writerows(top_features_records)

print(f'Top features saved to {tf_csv}')
print()
display_top_features_comparison(
    feature_sets=top_features_by_question,
    scores_sets=top_scores_by_question,
    neuronpedia_model='gemma-2-2b',
)

---

## Section 4 — Cross-Question Feature Recurrence

A feature that appears in the top-`TOP_N` list for many different questions is a candidate for a
**shared circuit component** — one that the model uses broadly for this type of reasoning.

We count how many questions each feature appears in. Features that appear in
`RECURRING_THRESHOLD` or more questions are flagged as recurring and become targets for
circuit interventions in Section 5.

In [19]:
feature_frequency: dict = defaultdict(list)

for qid, features in top_features_by_question.items():
    for feat in features:
        feature_frequency[tuple(feat)].append(qid)

sorted_features = sorted(feature_frequency.items(), key=lambda x: len(x[1]), reverse=True)

freq_records = []
for feat, qids in sorted_features:
    layer, pos, feat_idx = feat
    freq_records.append({
        'run_id':       run_id,
        'layer':        layer,
        'position':     pos,
        'feature_idx':  feat_idx,
        'frequency':    len(qids),
        'question_ids': ','.join(sorted(qids)),
    })

ff_csv = TABLES_DIR / 'feature_frequency.csv'
with open(ff_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(freq_records[0].keys()))
    writer.writeheader()
    writer.writerows(freq_records)

print(f'Unique features across all top-{TOP_N} lists: {len(sorted_features)}')
print(f'Feature frequency saved to {ff_csv}')
print()
print(f"{'Feature (layer, pos, feat_idx)':<36} {'Freq':>5}  Questions")
print('-' * 75)
for feat, qids in sorted_features:
    if len(qids) < 2:
        break
    layer, pos, feat_idx = feat
    print(f'({layer:>3}, {pos:>3}, {feat_idx:>8})                 {len(qids):>4}  {", ".join(sorted(qids))}')

---

## Section 5 — Circuit Interventions

Attribution graphs show correlation, not causation. To claim that a feature *causes* the
classification decision, we perturb it and check that the output changes in the predicted direction.

For each question, we apply two interventions to the top recurring features:

- **Amplify (5x)** — scale the feature's activation to five times its natural value.
  If it pushes toward `True`, amplification should increase the True-False margin.
- **Ablate (0x)** — set the feature's activation to zero.
  If it was supporting `True`, ablation should decrease the margin or flip the prediction.

We record `margin_before`, `margin_after`, and `delta_margin`. A consistent directional effect
across multiple questions supports a causal interpretation.

In [20]:
RECURRING_THRESHOLD = 3
TOP_K_INTERV = 10
TOP_K_SINGLE = 3   # per-feature tests for top aligned candidates

recurring_features = [feat for feat, qids in sorted_features if len(qids) >= RECURRING_THRESHOLD][:TOP_K_INTERV]

print(f'Recurring features (freq >= {RECURRING_THRESHOLD}): {len(recurring_features)}')
for f in recurring_features:
    layer, pos, feat_idx = f
    freq = len(feature_frequency[f])
    print(f'  ({layer}, {pos}, {feat_idx}) — appears in {freq} questions')

In [21]:
intervention_records = []

def _run_intervention(input_ids, features, scale, activations):
    """Apply a single intervention and return the post-intervention logits."""
    if not features:
        return model.get_activations(input_ids, sparse=True)[0]
    tuples = [
        (layer, pos, feat_idx, scale * activations[layer, pos, feat_idx])
        for (layer, pos, feat_idx) in features
    ]
    logits, _ = model.feature_intervention(input_ids, tuples)
    return logits


def _record(qid, label, margin_before, interv_type, scope, direction, feature_key, interv_logits):
    last_i       = interv_logits.squeeze(0)[-1]
    margin_after = (last_i[IDX_TRUE] - last_i[IDX_FALSE]).item()
    delta        = margin_after - margin_before
    intervention_records.append({
        'run_id':           run_id,
        'id':               qid,
        'label':            label,
        'intervention':     interv_type,
        'scope':            scope,
        'direction':        direction,
        'feature_key':      feature_key,
        'margin_before':    round(margin_before, 4),
        'margin_after':     round(margin_after,  4),
        'delta_margin':     round(delta, 4),
        'predicted_before': margin_before > 0,
        'predicted_after':  margin_after  > 0,
        'flipped':          (margin_before > 0) != (margin_after > 0),
    })


for q in QUESTIONS:
    qid       = q['id']
    label     = q['label']
    prompt    = format_prompt(q, ACTIVE_TEMPLATE)
    input_ids = model.ensure_tokenized(prompt)

    original_logits, activations = model.get_activations(input_ids, sparse=True)
    last          = original_logits.squeeze(0)[-1]
    margin_before = (last[IDX_TRUE] - last[IDX_FALSE]).item()

    active_set = {tuple(f.tolist()) for f in graphs[qid].active_features}
    cands      = [f for f in recurring_features if tuple(f) in active_set]

    # Build per-feature signed scores for this question
    feat_score = {}
    for feat, score in zip(top_features_by_question.get(qid, []),
                           top_scores_by_question.get(qid,   [])):
        feat_score[tuple(feat)] = score

    # Sign-aware split: pro_true features push margin positive, pro_false push negative
    pro_true  = [f for f in cands if feat_score.get(tuple(f), 0) > 0]
    pro_false = [f for f in cands if feat_score.get(tuple(f), 0) < 0]

    # aligned = features whose natural effect matches the question's label
    aligned = pro_true  if label else pro_false
    opposed = pro_false if label else pro_true

    # ── Group interventions (4 combinations) ─────────────────────────────────
    for direction, group in [('aligned', aligned), ('opposed', opposed)]:
        for interv_type, scale in [('amplify_5x', 5.0), ('ablate', 0.0)]:
            logits = _run_intervention(input_ids, group, scale, activations)
            _record(qid, label, margin_before, interv_type, 'group', direction, None, logits)
            cleanup_cuda()

    # ── Single-feature interventions on top-K aligned features ───────────────
    # Sort aligned by absolute score magnitude, take top TOP_K_SINGLE
    top_aligned = sorted(aligned, key=lambda f: abs(feat_score.get(tuple(f), 0)), reverse=True)[:TOP_K_SINGLE]
    for feat in top_aligned:
        fkey = str(feat)
        for interv_type, scale in [('amplify_5x', 5.0), ('ablate', 0.0)]:
            logits = _run_intervention(input_ids, [feat], scale, activations)
            _record(qid, label, margin_before, interv_type, 'single', 'aligned', fkey, logits)
            cleanup_cuda()


if intervention_records:
    interv_csv = TABLES_DIR / 'interventions.csv'
    with open(interv_csv, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(intervention_records[0].keys()))
        writer.writeheader()
        writer.writerows(intervention_records)
    print(f'Interventions complete. {len(intervention_records)} records saved to {interv_csv}')
else:
    print('No intervention records produced (check recurring_features and active_set overlap).')

---

## Section 6 — Summary Tables and Metrics

Four consolidated tables summarise the run. A `summary_metrics.json` file encodes the
aggregated metrics and is the primary file the analysis notebook uses for high-level overview.

In [22]:
print('=' * 55)
print('TABLE 1 — BASELINE ACCURACY BY TEMPLATE')
print('=' * 55)
accuracy_by_template = {}
for tkey in TEMPLATES:
    recs      = [r for r in baseline_records if r['template'] == tkey]
    n_correct = sum(1 for r in recs if r['correct'])
    pct       = 100 * n_correct / len(recs)
    accuracy_by_template[tkey] = {'correct': n_correct, 'total': len(recs), 'pct': round(pct, 1)}
    print(f'  {tkey}: {n_correct}/{len(recs)} correct ({pct:.0f}%)')

print()
print('=' * 55)
print(f'TABLE 2 — BASELINE MARGINS, TEMPLATE {ACTIVE_TEMPLATE}')
print('=' * 55)
print(f"{'ID':<8} {'Label':<6} {'Margin':>8}  Result")
print('-' * 32)
for r in [x for x in baseline_records if x['template'] == ACTIVE_TEMPLATE]:
    label_str = 'True ' if r['label'] else 'False'
    ok_str    = 'OK' if r['correct'] else 'WRONG'
    print(f"{r['id']:<8} {label_str:<6} {r['margin']:>8.4f}  {ok_str}")

print()
print('=' * 55)
print('TABLE 3 — TOP RECURRING FEATURES')
print('=' * 55)
print(f"{'(layer, pos, feat_idx)':<32} {'Freq':>5}")
print('-' * 38)
for feat, qids in sorted_features[:10]:
    print(f'{str(feat):<32} {len(qids):>5}')

print()
print('=' * 55)
print('TABLE 4 — INTERVENTION SUMMARY')
print('=' * 55)

def _interv_stats(recs):
    if not recs:
        return None
    avg_delta = sum(r['delta_margin'] for r in recs) / len(recs)
    n_flipped = sum(1 for r in recs if r['flipped'])
    return {'avg_delta_margin': round(avg_delta, 4), 'n_flipped': n_flipped, 'total': len(recs)}

interv_summary = {}
for group_label, scope, direction in [
    ('group_aligned', 'group',  'aligned'),
    ('group_opposed', 'group',  'opposed'),
    ('single_aligned', 'single', 'aligned'),
]:
    for interv_type in ['amplify_5x', 'ablate']:
        recs = [r for r in intervention_records
                if r['scope'] == scope and r['direction'] == direction
                and r['intervention'] == interv_type]
        stats = _interv_stats(recs)
        if stats:
            key = f'{group_label}_{interv_type}'
            interv_summary[key] = stats
            print(f'  {key:<34}: avg delta = {stats["avg_delta_margin"]:+.4f}   flipped: {stats["n_flipped"]}/{stats["total"]}')

summary = {
    'run_id':               run_id,
    'model':                MODEL_NAME,
    'transcoder':           TRANSCODER_NAME,
    'backend':              BACKEND,
    'active_template':      ACTIVE_TEMPLATE,
    'n_questions':          len(QUESTIONS),
    'accuracy_by_template': accuracy_by_template,
    'n_recurring_features': len(recurring_features),
    'intervention_summary': interv_summary,
}
(TABLES_DIR / 'summary_metrics.json').write_text(json.dumps(summary, indent=2))
print(f'\nSummary metrics saved to {TABLES_DIR / "summary_metrics.json"}')

---

## Section 7 — Write Manifest

The **`run_config_snapshot.json`** file (question bank, model, attribution params) was already
written at the start of the run; it is the **exact** string used to compute `run_id`.

This cell writes **`manifest.json`** with git SHA, wall-clock time, and intervention hyperparameters
(`top_n`, `recurring_threshold`, etc.) that are not part of the fingerprint.

In [23]:
def _git_sha():
    try:
        return subprocess.check_output(
            ['git', '-C', str(_root), 'rev-parse', '--short', 'HEAD'],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
    except Exception:
        return 'unknown'


manifest = {
    'run_id':                    run_id,
    'timestamp_utc':             datetime.now(timezone.utc).isoformat(),
    'git_sha':                   _git_sha(),
    'notebook':                  '1_run_attribution.ipynb',
    'model_name':                MODEL_NAME,
    'transcoder_name':           TRANSCODER_NAME,
    'backend':                   BACKEND,
    'dtype':                     MODEL_DTYPE,
    'active_template':           ACTIVE_TEMPLATE,
    'n_questions':               len(QUESTIONS),
    'attr_batch_size':           ATTR_KWARGS['batch_size'],
    'attr_max_feature_nodes':    ATTR_KWARGS['max_feature_nodes'],
    'attr_offload':              ATTR_KWARGS['offload'],
    'recurring_threshold':       RECURRING_THRESHOLD,
    'top_k_intervention':        TOP_K_INTERV,
    'top_n_features':            TOP_N,
    'config_path':               str(CONFIG_PATH),
    'results_root':              str(RESULTS_ROOT),
    'run_config_snapshot':       'run_config_snapshot.json',
    'fingerprint_config_hash12': CONFIG_HASH,
    'fingerprint_version':       FINGERPRINT_VERSION,
}

(RESULTS_ROOT / 'manifest.json').write_text(json.dumps(manifest, indent=2))

print(f'Manifest written  : {RESULTS_ROOT / "manifest.json"}')
print(f'run_id            : {run_id}')
print(f'config hash (x12) : {CONFIG_HASH}')
print(f'git_sha           : {manifest["git_sha"]}')

---

## Section 8 — Graph Export and Viewer

Raw `.pt` graphs are converted to the JSON format expected by the interactive viewer bundled
with `circuit-tracer`. All graphs are exported so the analysis notebook can reference any of them.

**Viewer interaction notes:**
- Click a node to see its feature description.
- Ctrl+click to pin a node to the subgraph pane.
- Hold G and click to group nodes into a labelled supernode.

On Runpod, the IFrame will not load because the pod's localhost is not directly accessible.
Enable port forwarding for the printed port and open the URL in your local browser.

import gc, torch
del model
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.mem_get_info() if torch.cuda.is_available() else "cpu")

In [24]:
for q in QUESTIONS:
    qid        = q['id']
    graph_path = GRAPHS_PT_DIR / f'{qid}_{ACTIVE_TEMPLATE}.pt'
    slug       = f'{run_id[:15]}-{qid.lower()}-{ACTIVE_TEMPLATE.lower()}'
    create_graph_files(
        graph_or_path=graph_path,
        slug=slug,
        output_path=str(GRAPHS_JSON_DIR),
        node_threshold=0.8,
        edge_threshold=0.98,
    )
    print(f'  Graph JSON exported: {slug}')

print(f'\nAll graph files written to {GRAPHS_JSON_DIR}')

In [25]:
PORT   = 8047
server = serve(data_dir=str(GRAPHS_JSON_DIR) + '/', port=PORT)
url    = f'http://localhost:{PORT}/index.html'
print(f'Graph viewer at: {url}')
print('On Runpod: enable port forwarding for this port and open the URL in your browser.')
display(IFrame(src=url, width='100%', height='800px'))

In [26]:
# server.stop()

---

## Section 9 — Git Sync Instructions

After this notebook finishes, commit the run outputs and push to GitHub so they are available
for local analysis. Run in the Runpod terminal (replace `<run_id>` with the actual value printed above):

```bash
cd /workspace/thesis_circuit_breaker
git add my_work/results/runs/<run_id>/
git commit -m "Add run <run_id>"
git push
```

Then on your local machine:

```bash
git pull
```

The analysis notebook (`2_analyse_results.ipynb`) will automatically find the new run directory.

> **Size note.** Raw `.pt` files are typically a few MB each. For a pilot run of 8 questions they
> are safe to commit. For larger runs (50+ questions), consider committing only `graphs_json/` and
> adding `graphs_pt/` to `.gitignore`.

In [ ]:
# Quick next-token diagnostic: top-5 overall + answer candidates
qid = QUESTIONS[0]["id"]  # change if needed, e.g. "TX-02"
q = next(x for x in QUESTIONS if x["id"] == qid)
prompt = TEMPLATES[ACTIVE_TEMPLATE].replace("{S}", q["statement"])

input_ids = model.ensure_tokenized(prompt)
with torch.no_grad():
    logits, _ = model.get_activations(input_ids)

last = logits.squeeze(0)[-1].float()
probs = torch.softmax(last, dim=-1)

print("Prompt:")
print(prompt)
print("-" * 80)

# Top-5 next-token predictions
print("Top-5 next tokens:")
topk = torch.topk(probs, k=5)
for rank, (tid, p) in enumerate(zip(topk.indices.tolist(), topk.values.tolist()), start=1):
    tok = tokenizer.decode([tid])
    logit = last[tid].item()
    print(f"{rank}. id={tid:6d}  token={tok!r:20}  prob={p:.6e}  logit={logit:+.4f}")

# Focused answer-candidate comparison
print("\nAnswer-candidate scores:")
cands = [" True", " False", "True", "False", " Yes", " No"]
for t in cands:
    ids = tokenizer.encode(t, add_special_tokens=False)
    if len(ids) != 1:
        print(f"{t!r:10} -> {ids} (skip: not single-token)")
        continue
    tid = ids[0]
    print(f"{t!r:10} id={tid:6d}  prob={probs[tid].item():.6e}  logit={last[tid].item():+.4f}")

# Current metric for notebook baseline
if len(tokenizer.encode(" True", add_special_tokens=False)) == 1 and len(tokenizer.encode(" False", add_special_tokens=False)) == 1:
    i_true = tokenizer.encode(" True", add_special_tokens=False)[0]
    i_false = tokenizer.encode(" False", add_special_tokens=False)[0]
    margin = last[i_true].item() - last[i_false].item()
    print(f"\nCurrent margin (' True' - ' False') = {margin:+.4f}")